# Descarga mensual de PPT y ETP (TerraClimate) para Sinaloa
Descarga **una imagen por mes y por variable**, para todos los meses disponibles en la colección, con el mismo flujo de la descarga por temporadas.

Archivos generados: `ppt_<año>_<mes>.tif` y `etp_<año>_<mes>.tif` (ej. `ppt_1958_01.tif`).

- Son valores **crudos** de TerraClimate: `pr` en mm y `pet` en décimas de mm (factor 0.1). Con `APLICAR_ESCALA = False` se guardan así, igual que en la descarga por temporadas; el factor se aplica después al leerlos.
- Se guardan en una carpeta distinta (`mapas_sinaloa_mensual`) para no mezclarlos con los archivos por temporada.
- Si se interrumpe, volver a correr: se saltan los archivos que ya existen.

In [1]:
# instalamos earth engine
# !pip install earthengine-api geemap

In [3]:
import os
import time
import ee
import geemap

ee.Initialize(project='...')
print("Google Earth Engine inicializado correctamente.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Google Earth Engine inicializado correctamente.


In [4]:
# Coordenadas BBox para Sinaloa: [longitud_min, latitud_min, longitud_max, latitud_max]
bbox_sinaloa = ee.Geometry.BBox(-109.4832, 22.4354, -105.3639, 27.0645)

# Carpeta local para almacenar las descargas mensuales
output_dir = "mapas_sinaloa_mensual"
os.makedirs(output_dir, exist_ok=True)

print(f"Carpeta de destino lista: '{output_dir}'")

Carpeta de destino lista: 'mapas_sinaloa_mensual'


In [5]:
COLECCION = "IDAHO_EPSCOR/TERRACLIMATE"

# Mapeo de nombres para archivos -> bandas oficiales en TerraClimate
VARIABLES = {
    "ppt": "pr",   # Precipitación (mm)
    "etp": "pet"   # Evapotranspiración Potencial (décimas de mm, factor 0.1)
}

# Rango de años: None = todos los meses disponibles en la colección
YEAR_START = None
YEAR_END = None

# False = valores crudos
# True  = multiplica por el factor de escala de cada banda al descargar
APLICAR_ESCALA = True
ESCALAS = {"pr": 1.0, "pet": 0.1}

# Reintentos ante errores de red / limites de Earth Engine
REINTENTOS = 3
ESPERA_SEG = 5

### Detección automática de los meses disponibles
Consulta a Earth Engine la primera y la última fecha de la colección, así que se actualiza solo cuando TerraClimate agrega meses nuevos.

In [6]:
coleccion = ee.ImageCollection(COLECCION)

t_ini = ee.Date(coleccion.aggregate_min("system:time_start")).format("YYYY-MM").getInfo()
t_fin = ee.Date(coleccion.aggregate_max("system:time_start")).format("YYYY-MM").getInfo()
y0, m0 = map(int, t_ini.split("-"))
y1, m1 = map(int, t_fin.split("-"))


def mes_siguiente(year, mes):
    return (year + 1, 1) if mes == 12 else (year, mes + 1)


meses = []
y, m = y0, m0
while (y, m) <= (y1, m1):
    if (YEAR_START is None or y >= YEAR_START) and (YEAR_END is None or y <= YEAR_END):
        meses.append((y, m))
    y, m = mes_siguiente(y, m)

print(f"Disponible en TerraClimate: {t_ini} a {t_fin}")
print(f"A descargar: {len(meses)} meses x {len(VARIABLES)} variables = {len(meses) * len(VARIABLES)} archivos "
      f"({meses[0][0]}-{meses[0][1]:02d} a {meses[-1][0]}-{meses[-1][1]:02d})")

Disponible en TerraClimate: 1958-01 a 2024-12
A descargar: 804 meses x 2 variables = 1608 archivos (1958-01 a 2024-12)


### Bucle principal de descarga
Una imagen por mes y variable, recortada a Sinaloa a 4000 m. Si una descarga falla, se reintenta y se elimina cualquier archivo incompleto; los que no se logren quedan listados al final.

In [7]:
descargados, saltados, fallidos = 0, 0, []

for year, mes in meses:
    str_inicio = f"{year}-{mes:02d}-01"
    y2, m2 = mes_siguiente(year, mes)
    str_fin = f"{y2}-{m2:02d}-01"      # filterDate excluye la fecha final

    for var_nombre, var_banda in VARIABLES.items():
        filename_tif = f"{var_nombre}_{year}_{mes:02d}.tif"
        filepath = os.path.join(output_dir, filename_tif)

        # Omitir descarga si el archivo ya fue descargado previamente
        if os.path.exists(filepath):
            saltados += 1
            continue

        print(f"Procesando y descargando: {filename_tif}...")

        for intento in range(1, REINTENTOS + 1):
            try:
                img = (
                    coleccion
                    .filterDate(str_inicio, str_fin)
                    .select(var_banda)
                    .first()
                )
                if APLICAR_ESCALA:
                    img = img.multiply(ESCALAS[var_banda])
                img = img.clip(bbox_sinaloa)

                geemap.ee_export_image(
                    img,
                    filename=filepath,
                    scale=4000,
                    region=bbox_sinaloa,
                    file_per_band=False
                )
                if not os.path.exists(filepath):
                    raise RuntimeError("Earth Engine no genero el archivo")
                descargados += 1
                break
            except Exception as e:
                if os.path.exists(filepath):
                    os.remove(filepath)          # evita dejar archivos corruptos
                print(f"  Intento {intento}/{REINTENTOS} fallido para {filename_tif}: {e}")
                if intento < REINTENTOS:
                    time.sleep(ESPERA_SEG)
        else:
            fallidos.append(filename_tif)

print(f"\nDescargados: {descargados} | Saltados (ya existian): {saltados} | Fallidos: {len(fallidos)}")
if fallidos:
    print("Vuelve a correr esta celda para reintentar:", fallidos[:10], "..." if len(fallidos) > 10 else "")
else:
    print("¡Proceso finalizado! Todos los mapas mensuales están disponibles en la carpeta local.")

Procesando y descargando: ppt_1958_01.tif...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\Leon1\Documents\Maestria\Semestre III\Movilidad\Tesis\mapas_sinaloa_mensual\ppt_1958_01.tif
Procesando y descargando: etp_1958_01.tif...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\Leon1\Documents\Maestria\Semestre III\Movilidad\Tesis\mapas_sinaloa_mensual\etp_1958_01.tif
Procesando y descargando: ppt_1958_02.tif...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\Leon1\Documents\Maestria\Semestre III\Movilidad\Tesis\mapas_sinaloa_mensual\ppt_1958_02.tif
Procesando y descargando: etp_1958_02.tif...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\Leon1\Documents\Maestria\Semestre III\Movilidad\Tesis\mapas_sinaloa_mensual\etp_1958_02.tif
Procesando y descargando: ppt_1958_03.tif...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\Leon1\Documents\Maestria\Semestre III\Movilidad\Tesis\mapas_sinaloa_mensual\ppt_1958_03.tif


### Verificación de los archivos
Comprueba que cada archivo esperado exista y se pueda abrir, y que todos tengan la misma malla (necesario para el cálculo del IA).

In [8]:
# !pip install rasterio
import rasterio

esperados = [f"{v}_{y}_{m:02d}.tif" for y, m in meses for v in VARIABLES]
faltan = [f for f in esperados if not os.path.exists(os.path.join(output_dir, f))]

corruptos, mallas = [], set()
for f in esperados:
    ruta = os.path.join(output_dir, f)
    if not os.path.exists(ruta):
        continue
    try:
        with rasterio.open(ruta) as src:
            mallas.add((src.width, src.height, src.crs.to_string() if src.crs else None, tuple(round(c, 6) for c in src.transform[:6])))
    except Exception:
        corruptos.append(f)

print(f"Esperados: {len(esperados)} | Faltan: {len(faltan)} | Corruptos: {len(corruptos)}")
print(f"Mallas distintas: {len(mallas)} (debe ser 1)")
if faltan:    print("Faltan (primeros 10):", faltan[:10])
if corruptos: print("Corruptos (borralos y vuelve a correr la descarga):", corruptos[:10])

Esperados: 1608 | Faltan: 0 | Corruptos: 0
Mallas distintas: 1 (debe ser 1)
